Import libraries, set date range for analysis period, and initialize a `Sources` class object - it contains all the stock/index name and the corresponding tickers that we need. We also initialize `data_frames` to compile all DataFrames.

In [1]:
import requests
from datetime import datetime, timedelta
import pandas as pd
from fredapi import Fred 
import yfinance as yf
from dotenv import load_dotenv
import os
import sys

sys.path.append('..')
from modules.source import Sources, RAW_DATA_PATH, PROCESSED_DATA_PATH, daily_master_csv, monthly_master_csv
from modules.helpers import clean_api_response
stock = Sources()

START = datetime(2015, 1, 1)
END = datetime(2026, 8, 6)
   
load_dotenv("../.env")
FRED_API_KEY = os.getenv('FRED_API_KEY')
if FRED_API_KEY is None: print('Enter your FRED API key in .env')
fred = Fred(FRED_API_KEY)

data_frames: dict[pd.DataFrame] = {}

Since these operations are identical across all tickers, from here on we only note material deviations from this structure.
1. **Fetch** - pull data from FRED
2. **Save** - write the raw response to `.csv`
3. **Process** - use `clean_api_response` to standardize data format, and adjust to add back publication lag (if any) 
4. **Add to `data_frames` object**

We convert the raw data to DataFrame before saving it purely for pipeline consistency. Saving it as a Series instead would produce an identical CSV.



In [2]:
series = fred.get_series_first_release(stock.palm_oil_global.ticker)

df = clean_api_response(series, stock.palm_oil_global)
df.index = pd.to_datetime(df.index)

df.to_csv(f"{RAW_DATA_PATH}/{stock.palm_oil_global}.csv")

df = df.loc[START:END]
data_frames[stock.palm_oil_global] = df

FFR is provided with a range, so we take the midpoint.

In [3]:
series_upper = fred.get_series('DFEDTARU')
series_lower = fred.get_series('DFEDTARL')

df = pd.concat([series_upper, series_lower], axis=1, join='inner')
df.columns = ['Upper', 'Lower']
df['Midpoint'] = (df['Upper'] + df['Lower']) / 2
df.index.name = 'date'
df.to_csv(f"{RAW_DATA_PATH}/{stock.FFR_midpoint}.csv")

df = df[['Midpoint']].rename(columns={'Midpoint': stock.FFR_midpoint})
df = df.loc[START:END]
data_frames[stock.FFR_midpoint] = df

The EFFR data is already adjusted for 1 day publication lag, so we shift it back to the publication date.

In [4]:
series = fred.get_series(stock.EFFR.ticker)
df.to_csv(f"{RAW_DATA_PATH}/{stock.EFFR}.csv")

df = clean_api_response(series, stock.EFFR)

lag = timedelta(days=1)
df = df.loc[START - lag : END - lag]
df = df.shift(1, lag)

data_frames[stock.EFFR] = df

We pull the data from yfinance, and use the daily closing price as our price data.

In [5]:
yahoo_stocks = [stock.UST_10Y, stock.VIX, stock.USDMYR, stock.DXY, stock.brent_oil, stock.KLCI]

for name in yahoo_stocks:
   df = yf.download(name.ticker, START, END, progress=False)
   df.to_csv(f'{RAW_DATA_PATH}/{name}.csv')

   try:
      df = clean_api_response(df, name)
   except ValueError as e:
      print(f"  FAILED - {name}: {e}")
      continue

   print(f"{name} ({name.ticker}) - saved {len(df)} rows")
   df = df[['Close']].rename(columns={'Close': name})
   data_frames[name] = df

UST_10Y (^TNX) - saved 2913 rows
VIX (^VIX) - saved 2915 rows
USDMYR (MYR=X) - saved 3018 rows
DXY (DX-Y.NYB) - saved 2915 rows
Brent_Oil (BZ=F) - saved 2915 rows
KLCI (^KLSE) - saved 2837 rows


For CPI inflation by DOSM, we narrow it down to overall division only and YoY inflation as our primary series. And undo the ~2 month of publication lag.

One thing to note: core CPI data is only available from 2018 onwards, so YoY (needing a 12-month lookback) is only available from 2019 onwards - this affects CPI-related analysis specifically, not the full project's date range.

In [6]:
df_cpi = pd.read_parquet('https://storage.dosm.gov.my/cpi/cpi_2d_core_inflation.parquet')
df_cpi.to_csv(f'{RAW_DATA_PATH}/CPI.csv')

df_cpi = (
   df_cpi
   .assign(date=pd.to_datetime(df_cpi['date']))
   .set_index('date')
   .query("division == 'overall'")
   .drop(columns=['division', 'inflation_mom'])
   .shift(2, freq='ME')
   .loc[START:END]
   .dropna(how='any')
   .rename(columns={'inflation_yoy': stock.cpi_inflation_yoy})
)

data_frames[stock.cpi_inflation_yoy] = df_cpi

As with core CPI, for OPR, we set `date` as the index, then sort it before saving. The year-by-year fetch doesn't guarantee chronological order.

Post-save, we drop `year` (redundant once `date` is the index) and `change_in_opr` (we'll reconstruct this ourselves downstream), and rename `new_opr_level` to `OPR`.

In [7]:
headers = {'Accept': 'application/vnd.BNM.API.v1+json'}

records = []
for year in range(START.year, END.year + 1):
   resp = requests.get(f'https://api.bnm.gov.my/public/opr/year/{year}', headers=headers)
   records.extend(resp.json()['data'])

df_opr = pd.DataFrame(records)
df_opr['date'] = pd.to_datetime(df_opr['date'])
df_opr = df_opr.set_index('date')
df_opr = df_opr.sort_index()

df_opr = df_opr[~df_opr.index.duplicated(keep='last')]
df_opr.to_csv(f'{RAW_DATA_PATH}/{stock.OPR}.csv')

df_opr = (
   df_opr.loc[START:END]
   .drop(columns=['year', 'change_in_opr'])
   .rename(columns={'new_opr_level': stock.OPR})
)

data_frames[stock.OPR] = df_opr

For each sector, we aggregate 2-3 large, widely-held constituents, save to CSV, then reconstruct its equal-weighted index. For plantation, we trim data from before 2017-11-30 (SD Gurthie Berhad hasn't listed yet) or else it would produce price-level break.

In [8]:
sector_constituents = {
   stock.financials: ['1155.KL', '1023.KL', '1295.KL'],   # Maybank, CIMB, Public Bank
   stock.plantation: ['5285.KL', '1961.KL', '2445.KL'],   # Sime Darby Plantation, IOI Corp, KLK
   stock.reits:      ['5227.KL', '5176.KL', '5235SS.KL'], # IGB REIT, Sunway REIT, KLCCP Stapled
   stock.technology: ['0166.KL', '0097.KL', '0128.KL'],   # Inari Amertron, ViTrox, Frontken
   stock.energy:     ['6033.KL', '5183.KL', '5681.KL'],   # PetGas, PetChem, Petronas Dagangan
   stock.industrial_products: ['8869.KL', '7113.KL'],     # Press Metal, Top Glove
}

for sector, tickers in sector_constituents.items():
   print(f"\n=== {sector.upper()} ===")
   closing_price_series: dict[pd.Series] = {}
   
   for t in tickers:
      df = yf.download(t, START, END, progress=False)
      try:
         df = clean_api_response(df, t)
      except ValueError as e:
         print(f"  FAILED - {t}: {e}")
         continue

      closing_price_series[t] = df['Close'] # stored as series
      print(f"  OK - {t} ({len(df)} rows)")

   sector_df = pd.DataFrame(closing_price_series)
   sector_df.to_csv(f'{RAW_DATA_PATH}/sectors/{sector}.csv')
   sector_df.index.name = 'date'

   if sector is stock.plantation:
      cutoff = datetime(2017, 11, 30) # 5285.KL (SD Guthrie Berhad) date of being listed
      sector_df = sector_df.loc[cutoff:]

   sector_df[sector] = sector_df.ffill().pct_change().mean(axis=1)
   sector_df[sector] = 100 * (1 + sector_df[sector].fillna(0)).cumprod()
   data_frames[sector] = sector_df[sector]


=== FINANCIALS ===
  OK - 1155.KL (2855 rows)
  OK - 1023.KL (2856 rows)
  OK - 1295.KL (2856 rows)

=== PLANTATION ===
  OK - 5285.KL (2129 rows)
  OK - 1961.KL (2855 rows)
  OK - 2445.KL (2855 rows)

=== REITS ===
  OK - 5227.KL (2854 rows)
  OK - 5176.KL (2855 rows)
  OK - 5235SS.KL (2855 rows)

=== TECHNOLOGY ===
  OK - 0166.KL (2855 rows)
  OK - 0097.KL (2856 rows)
  OK - 0128.KL (2856 rows)

=== ENERGY ===
  OK - 6033.KL (2855 rows)
  OK - 5183.KL (2855 rows)
  OK - 5681.KL (2856 rows)

=== INDUSTRIAL_PRODUCTS ===
  OK - 8869.KL (2855 rows)
  OK - 7113.KL (2855 rows)


Now we adjust for timezone difference (look-ahead bias).

In [9]:
us_market_ticker = [stock.EFFR, stock.UST_10Y, stock.USDMYR, stock.DXY, stock.VIX, stock.brent_oil, stock.palm_oil_global]

for i, df in data_frames.items():
   df = df.ffill()
   lag = timedelta(days=1)
   if i in us_market_ticker:
      df = df.shift(-1, lag).loc[START:]
   data_frames[i] = df.loc[START:END - lag]

Combine to master DataFrame and save to CSV.

In [10]:
daily_master: pd.DataFrame = pd.concat(data_frames.values(), axis=1, sort=True)
daily_master.to_csv(f"{PROCESSED_DATA_PATH}/{daily_master_csv}")

In [11]:
monthly_master = daily_master.ffill().resample("ME").last()
monthly_master.to_csv(f"{PROCESSED_DATA_PATH}/{monthly_master_csv}")